# Forecast Desk — one session, end to end

`asof = 2025-05-29`. The deterministic forecast is assumed cached
(`python -m src.forecasts --asof 2025-05-29`). This notebook walks the loop:

1. the no-lookahead price cut Kronos sees
2. the sampled path distribution (dispersion restored) + fan chart
3. cross-sectional ranking and the per-name strength z-score
4. the triage agent: setup → analogs → cluster → memory → keep/drop → brief
5. the groundedness critic
6. the close-of-day grade
7. the post-mortem: classify *why*, write a desk-memory episode


In [ ]:
import os
if "src" not in os.listdir("."):
    os.chdir("..")

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from src import config
ASOF = '2025-05-29'
DEV = config.DEV_UNIVERSE
ASOF

## 1 · The no-lookahead cut

In [ ]:
from src.data import load_context
pl = load_context("NVDA", ASOF)
print("source:", pl.source, "| context through:", pl.last_context_date.date(),
      "| bars:", len(pl.df))
assert pl.df.index.max() < pd.Timestamp(ASOF)   # strictly before asof
pl.df.tail()

## 2 · Kronos with dispersion + fan chart

In [ ]:
from src.kronos_infer import forecast, fan_chart
fc = forecast(pl)
print(f"median {fc.median_ret*100:+.2f}%  P(up) {fc.p_up:.0%}  "
      f"dispersion {fc.std_ret*100:.2f}%  strength {fc.strength:+.2f}")
fan_chart(fc); plt.show()

## 3 · Cross-sectional rank + strength-z

In [ ]:
from src.features import rank_table
from src.forecasts import strength_history
from src.store import load
f = load("forecasts"); f = f[f["asof"].astype(str) == ASOF]
hist = {t: strength_history(t, ASOF) for t in f["ticker"]}
rt = rank_table(f[["ticker","q50","q05","q95","p_up","std","strength"]], strength_hist=hist)
rt.head(12)

## 4 · Triage agent

In [ ]:
from src.triage import run as run_triage
st = run_triage(ASOF)
print("\n".join(st["trace"]))
print("\nkept:", st["kept"])

In [ ]:
print(open(config.LOG_DIR / f"brief_{ASOF}.md").read())

## 5 · Groundedness critic (on one brief)

In [ ]:
from src.critic import review
from src.triage import _flat_values
t = st["kept"][0]
rep = review(st["briefs"][t]["brief_md"], _flat_values(st["dossiers"][t]),
             setup=st["dossiers"][t]["setup"])
print(f"{t}: groundedness {rep.groundedness:.0%}, ok={rep.ok}, "
      f"ungrounded={rep.ungrounded}, issues={rep.llm_issues}")

## 6 · Close-of-day grade

In [ ]:
from src.grade import run as run_grade, report
graded = run_grade(ASOF)
pd.DataFrame(graded)[["ticker","ret","actual_quantile","inside_envelope","dir_match"]]

## 7 · Post-mortem → desk memory

In [ ]:
from src.postmortem import run as run_pm
pm = run_pm(ASOF)
print("\n".join(pm["trace"]))

In [ ]:
from src.memory import recall
for t in st["kept"]:
    s = st["dossiers"][t]["setup"]
    print(recall(t, s).line())

---
The memory line above is what the **next** morning's triage reads back for these
names. Run `notebooks/loop_replay.ipynb` to watch it accumulate over the window.